# CRIC-only controlled SSL comparison: final protocol v3

This notebook runs the agreed CRIC comparison and writes only to `data/results/cric_final_protocol_v3/`.


## Arms and run counts
- Supervised-only, standard augmentation: 5% and 10%, seeds 42–44 (6 runs).
- Supervised-only, same strong augmentation as the SSL strong view: 5% and 10%, seeds 42–44 (6 runs).
- Supervised-only with 100% of CRIC training labels: seeds 42–44 (3 runs).
- FixMatch at τ=0.95: 5% and 10%, seeds 42–46 (10 runs).
- FixMatch at τ=0.80 with no gates: 5% and 10%, seeds 42–44 (6 runs).
- CAS-Match: fixed τ_low=0.80, τ_high=0.95, agreement AND stability gates on low band; 5% and 10%, seeds 42–46 (10 runs).

**Total: 41 runs.** Seeds 42–44 cover every arm and budget. Seeds 45–46 extend only the primary FixMatch-vs-CAS-Match comparison. The queue runs seed-by-seed, so each seed's full paired comparison finishes before the next seed starts.

## Protocol locked before training
1. **Equal optimizer-step budget:** for 5%/10% supervised arms, one epoch has exactly `ceil(number_of_unlabeled_training_images / batch_size)` optimizer steps, matching the paired SSL arm. The labeled subset is sampled with replacement/cycled to fill those steps. The 100%-label supervised reference uses one full labeled-training pass per epoch.
2. **Model selection:** EMA weights (decay 0.995) are evaluated for every method. Select using the rolling mean of the latest 3 EMA validation Macro-F1 values. No checkpoint is eligible before epoch 8. Early-stopping patience starts only after epoch 8.
3. **Pseudo-label inference mode:** weak/strong pseudo-label predictions are generated with the student in evaluation mode, disabling stochastic depth during the admission decision. The student is returned to training mode for gradient updates. This avoids stochastic-depth noise contaminating agreement and confidence.
4. **Fixed thresholds:** FixMatch admits confidence ≥0.95. FixMatch-0.80 admits confidence ≥0.80 without gates. CAS-Match admits ≥0.95 automatically and admits the 0.80–0.95 band only when weak/strong agreement AND temporal stability pass. No validation-fitted τ_low or silent fallback is used.
5. **Diagnostics:** log candidate, accepted and rejected counts/correctness and true-class counts for every predicted class, confidence band (high, low, below-low), and agreement/stability combination. Acceptance rate uses the entire unlabeled pool, including below-low candidates.
6. **Test artifacts:** save test probabilities for every run. Report Macro-F1, AUROC, AUPRC, abnormal recall, specificity, balanced accuracy, and abnormal recall at a validation-selected 90% specificity threshold.
7. **Split integrity:** record and check hashes for each seed/budget labeled subset, validation split, test split and full manifest before training.
8. **Storage:** after test metrics, probabilities and `DONE.json` are safely written, delete student/EMA/optimizer checkpoints and retain histories, diagnostics, probabilities, protocol audit and metrics. Incomplete runs retain resumable checkpoints.

## Pre-registered interpretation rules
- Primary endpoint: test Macro-F1. Primary comparison: CAS-Match minus FixMatch-0.95, separately at 5% and 10%.
- For the five-seed primary comparison, call a method a **win** only if mean Macro-F1 gain is at least 1.0 percentage point and the gain has the same positive sign in at least 4 of 5 seeds. Apply the mirrored rule for a loss.
- For three-seed controls, require at least a 1.0-point mean gain and the same sign in all 3 seeds to call a win/loss.
- Call methods **approximately equivalent (≈)** when the absolute mean Macro-F1 gap is below 1.0 percentage point. If the mean gap is ≥1 point but the seed-direction rule is not met, call the result inconclusive.
- Report per-seed deltas, mean and standard deviation. Do not use p-values from three or five seeds as the main evidence. Test-image bootstrap intervals within each seed may be reported as a supplementary, conditional uncertainty analysis.
- Compare CAS-Match against FixMatch-0.80 to isolate whether the gates add value beyond lowering the confidence threshold. Compare FixMatch and CAS-Match against both standard and strong-augmentation supervised controls.

## Resource and timing preflight
Batch size 8, workers 0, pinned memory disabled, cleanup between epochs, disk guard at 8 GB, atomic checkpoint writes and previous-checkpoint fallback. Run the optional one-epoch FixMatch timing benchmark before committing to the full queue. After a completed run, bulky checkpoints are removed automatically.

**Do not launch the full queue until the smoke tests pass, split audits pass, and the timing benchmark is acceptable on your machine.**


# Previous notebook replaced

This notebook supersedes the earlier CAS-Match 20-run notebook for the new CRIC-only final protocol. Do not run the old queue or reuse its checkpoints.

In [1]:
# 1. Imports and configuration
from pathlib import Path
import os, json, time, random, hashlib, math, copy, warnings
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix, balanced_accuracy_score)

SEED = 42  # default; run-specific seed is passed into each experiment
DATASETS = ("cric",)
FRACTIONS = (0.05, 0.10)
METHODS = ("supervised", "supervised_strong", "supervised_full",
           "fixmatch", "fixmatch_tau80", "cas_match")
SEEDS_CORE = (42, 43, 44)
SEEDS_PRIMARY = (42, 43, 44, 45, 46)
TAU_LOW_FIXED = 0.80
EMA_DECAY = 0.995
VAL_SMOOTH_WINDOW = 3
MIN_SELECT_EPOCH = 8
TARGET_SPECIFICITY = 0.90
IMAGE_SIZE = 128
BATCH_SIZE = 8
MAX_EPOCHS = 60
PATIENCE = 10
LR = 1e-4
WEIGHT_DECAY = 1e-4
TAU_HIGH = 0.95
TAU_CANDIDATES = (0.80,)  # retained for metadata only; no validation-fitted threshold
PRECISION_TARGET = 0.90  # not used for threshold selection in this protocol
MIN_CALIBRATION_SUPPORT = 30  # legacy metadata only; no calibration is performed
MEMORY_K = 5
MEMORY_MIN = 3
STABILITY_MIN = 0.80
UNSUP_WEIGHT = 1.0
NUM_WORKERS = 0
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP = DEVICE.type == "cuda"

# Set ROOT explicitly if automatic discovery fails.
ROOT = None
_candidates = [Path.cwd(), Path.cwd().parent, Path.home() / "Documents" / "cervical-semi-sl"]
if ROOT is None:
    ROOT = next((p.resolve() for p in _candidates if (p / "data" / "splits_v2" / "cric_splits_v2.csv").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Project root not found. Set ROOT to your cervical-semi-sl folder.")
ROOT = Path(ROOT)
SPLIT_DIR = ROOT / "data" / "splits_v2"
OUT_DIR = ROOT / "data" / "results" / "cric_final_protocol_v3"  # separate from Phase 11 and prior CAS-Match runs
OUT_DIR.mkdir(parents=True, exist_ok=True)

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

CONFIG = {"seed":SEED, "datasets":DATASETS, "fractions":FRACTIONS, "methods":METHODS,
          "backbone":"convnext_tiny", "image_size":IMAGE_SIZE, "batch_size":BATCH_SIZE,
          "num_workers":NUM_WORKERS, "amp_enabled":AMP,
           "max_epochs":MAX_EPOCHS, "patience":PATIENCE, "lr":LR,
          "seeds_core":SEEDS_CORE, "seeds_primary":SEEDS_PRIMARY,
          "weight_decay":WEIGHT_DECAY, "tau_high":TAU_HIGH,
           "tau_low_fixed":TAU_LOW_FIXED, "ema_decay":EMA_DECAY,
          "val_smooth_window":VAL_SMOOTH_WINDOW, "min_select_epoch":MIN_SELECT_EPOCH,
           "target_specificity":TARGET_SPECIFICITY,
          "training_step_policy":"supervised_low_label_matches_unlabeled_batches_per_epoch_with_same_seeded_labeled_sampling; full_supervised_one_labeled_pass",
          "pseudo_label_inference_mode":"eval_disable_stochastic_depth",
          "memory_k":MEMORY_K,
          "memory_min":MEMORY_MIN, "stability_min":STABILITY_MIN,
          "unsup_weight":UNSUP_WEIGHT}
# Include augmentation choices in the resume identity. The split-file hashes are
# added after manifests are loaded in Cell 3.
AUGMENTATION_CONFIG = {
    "weak": {"horizontal_flip_p": 0.5, "rotation_degrees": 8},
    "strong": {"random_resized_crop_scale": (0.80, 1.0), "ratio": (0.90, 1.10),
               "color_jitter": (0.15, 0.15, 0.10, 0.02), "cutout_fraction": 0.25},
    "normalization_mean": MEAN, "normalization_std": STD,
}
CONFIG["augmentations"] = AUGMENTATION_CONFIG
CONFIG_HASH = hashlib.sha256(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()[:16]
print("ROOT:", ROOT, "\nOutput:", OUT_DIR, "\nDevice:", DEVICE, "AMP:", AMP)
print("Configuration hash:", CONFIG_HASH)

# ---- Conservative runtime and storage safeguards ----
import gc, shutil, tempfile, signal
RAM_BATCH_SIZE = BATCH_SIZE
MIN_FREE_GB = 8.0  # refuse to start an epoch if less space is available
CHECKPOINT_KEEP_BACKUP = True

def resource_report():
    try:
        import psutil
        vm=psutil.virtual_memory()
        print(f"RAM: available={vm.available/1024**3:.2f} GB / total={vm.total/1024**3:.2f} GB")
    except Exception as e:
        print("RAM report unavailable:", repr(e))
    du=shutil.disk_usage(OUT_DIR)
    print(f"Disk at output: free={du.free/1024**3:.2f} GB / total={du.total/1024**3:.2f} GB")
    if torch.cuda.is_available():
        print(f"CUDA allocated={torch.cuda.memory_allocated()/1024**3:.2f} GB; reserved={torch.cuda.memory_reserved()/1024**3:.2f} GB")

def assert_disk_headroom(required_gb=MIN_FREE_GB):
    free=shutil.disk_usage(OUT_DIR).free/1024**3
    if free < required_gb:
        raise RuntimeError(f"STOPPED SAFELY: only {free:.2f} GB free; need {required_gb:.1f} GB. Free space before running.")


ROOT: C:\Users\nanda\Documents\cervical-semi-sl 
Output: C:\Users\nanda\Documents\cervical-semi-sl\data\results\cric_final_protocol_v3 
Device: cuda AMP: True
Configuration hash: 74007c92a0977d65


## 1. Split manifests and seed-specific labeled subsets

The CRIC V2 split manifest is fixed. Each seed/budget gets one stratified labeled subset, stored once and reused across every method. A protocol audit records SHA-256 hashes for the subset, validation split, test split, and full manifest. If any hash changes, the run stops.

In [2]:
def resolve_crop(p):
    p = Path(str(p))
    candidates = [p] if p.is_absolute() else [ROOT / p, ROOT / "data" / p]
    for c in candidates:
        if c.exists(): return c.resolve()
    return None

manifests = {}
for ds in DATASETS:
    csv_path = SPLIT_DIR / f"{ds}_splits_v2.csv"
    if not csv_path.exists(): raise FileNotFoundError(csv_path)
    df = pd.read_csv(csv_path)
    required = {"crop_path", "binary_label", "split"}
    if required - set(df.columns): raise ValueError(f"{ds}: missing {required-set(df.columns)}")
    df = df.copy()
    df["split"] = df["split"].astype(str).str.strip().str.lower().replace({"validation":"val", "valid":"val"})
    if set(df["split"].unique()) != {"train", "val", "test"}:
        raise ValueError(f"{ds}: expected train/val/test, got {sorted(df.split.unique())}")
    df["binary_label"] = pd.to_numeric(df["binary_label"], errors="raise").astype(int)
    if not set(df.binary_label.unique()).issubset({0,1}): raise ValueError("Labels must be 0/1")
    df["resolved_path"] = df.crop_path.map(resolve_crop)
    if df.resolved_path.isna().any(): raise FileNotFoundError(f"Unresolved paths in {ds}: {df[df.resolved_path.isna()].crop_path.head().tolist()}")
    if df.resolved_path.duplicated().any(): raise ValueError(f"Duplicate crop paths in {ds}")
    manifests[ds] = df.reset_index(drop=True)
    print(f"\n{ds.upper()} rows={len(df):,}")
    display(pd.crosstab(df.split, df.binary_label, margins=True))

# Resume must not silently reuse checkpoints if either split manifest changes.
_manifest_hashes = {ds: hashlib.sha256((SPLIT_DIR / f"{ds}_splits_v2.csv").read_bytes()).hexdigest()
                    for ds in DATASETS}
CONFIG["manifest_sha256"] = _manifest_hashes
CONFIG_HASH = hashlib.sha256(json.dumps(CONFIG, sort_keys=True).encode()).hexdigest()[:16]
print("Final resume configuration hash (includes augmentations + split manifests):", CONFIG_HASH)

def seed_all(seed=SEED):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def labeled_frames(ds, frac, seed=SEED):
    train = manifests[ds].query("split == 'train'").reset_index(drop=True)
    subset_dir = OUT_DIR / "label_subsets"; subset_dir.mkdir(parents=True, exist_ok=True)
    pct = int(round(frac*100))
    subset_path = subset_dir / f"{ds}_labeled_{pct}pct_seed{seed}.csv"
    if subset_path.exists():
        saved = pd.read_csv(subset_path)
        wanted = set(saved.resolved_path.astype(str))
        lab = train[train.resolved_path.astype(str).isin(wanted)].copy()
        if len(lab) != len(wanted): raise ValueError(f"Saved subset mismatch: {subset_path}")
        if set(lab.binary_label.astype(int)) != set(saved.binary_label.astype(int)):
            raise ValueError(f"Saved subset labels changed: {subset_path}")
    else:
        rng = np.random.default_rng(seed + (0 if ds == "cric" else 10000) + pct)
        picked=[]
        for cls in (0,1):
            idx=np.flatnonzero(train.binary_label.to_numpy()==cls)
            n=len(idx) if frac >= 1.0 else max(1,int(round(frac*len(idx))))
            picked.extend(rng.choice(idx,size=min(n,len(idx)),replace=False).tolist())
        lab=train.iloc[sorted(picked)].copy()
        lab[["resolved_path","binary_label"]].to_csv(subset_path,index=False)
    unlab=train[~train.resolved_path.isin(lab.resolved_path)].copy()
    return lab.reset_index(drop=True), unlab.reset_index(drop=True)

def sha256_frame(frame, columns):
    payload=frame[columns].sort_values(columns).to_csv(index=False).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

def protocol_split_audit(ds, frac, seed, method_names):
    lab,_=labeled_frames(ds,frac,seed)
    val=manifests[ds].query("split == 'val'").reset_index(drop=True)
    test=manifests[ds].query("split == 'test'").reset_index(drop=True)
    hashes={
      "labeled_subset_sha256":sha256_frame(lab,["resolved_path","binary_label"]),
      "validation_sha256":sha256_frame(val,["resolved_path","binary_label"]),
      "test_sha256":sha256_frame(test,["resolved_path","binary_label"]),
      "train_manifest_sha256":_manifest_hashes[ds]
    }
    audit_dir=OUT_DIR/"protocol_audits"; audit_dir.mkdir(exist_ok=True)
    row={"dataset":ds,"fraction":frac,"seed":seed,**hashes,
         "methods":";".join(method_names)}
    audit_path=audit_dir/f"{ds}_{int(frac*100)}pct_seed{seed}.json"
    if audit_path.exists():
        prior=json.loads(audit_path.read_text())
        for k in hashes:
            if prior.get(k)!=hashes[k]:
                raise RuntimeError(f"Protocol hash mismatch for {k}: {audit_path}")
    else:
        audit_path.write_text(json.dumps(row,indent=2))
    return row

for ds in DATASETS:
    for frac in FRACTIONS:
        for seed in SEEDS_CORE:
            lab, unlab = labeled_frames(ds, frac, seed)
            print(f"{ds.upper()} {frac:.0%} seed={seed}: labeled={len(lab):,}, unlabeled={len(unlab):,}")



CRIC rows=11,534


binary_label,0,1,All
split,,,
test,1603,704,2307
train,4482,3592,8074
val,694,459,1153
All,6779,4755,11534


Final resume configuration hash (includes augmentations + split manifests): 5b290d3b91ffdd36
CRIC 5% seed=42: labeled=404, unlabeled=7,670
CRIC 5% seed=43: labeled=404, unlabeled=7,670
CRIC 5% seed=44: labeled=404, unlabeled=7,670
CRIC 10% seed=42: labeled=807, unlabeled=7,267
CRIC 10% seed=43: labeled=807, unlabeled=7,267
CRIC 10% seed=44: labeled=807, unlabeled=7,267


## 2. Augmentation controls

The standard supervised arm uses the weak geometric augmentation. The strong-augmentation supervised arm uses the exact strong transform used for FixMatch/CAS-Match strong views: random resized crop, color jitter, and cutout. All arms use the same resize and ImageNet normalization.

In [3]:
class Cutout:
    def __init__(self, fraction=0.25): self.fraction = fraction
    def __call__(self, tensor):
        # Applied after ToTensor and before normalization only in the strong transform.
        _, h, w = tensor.shape
        ch, cw = max(1, int(h*self.fraction)), max(1, int(w*self.fraction))
        top = random.randint(0, max(0, h-ch)); left = random.randint(0, max(0, w-cw))
        out = tensor.clone(); out[:, top:top+ch, left:left+cw] = 0.0
        return out

weak_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(8),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD),
])
strong_tf = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.80, 1.0), ratio=(0.90, 1.10)),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.10, hue=0.02),
    transforms.ToTensor(), Cutout(0.25), transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class CellDataset(Dataset):
    def __init__(self, frame, mode): self.frame=frame.reset_index(drop=True); self.mode=mode
    def __len__(self): return len(self.frame)
    def __getitem__(self, i):
        r=self.frame.iloc[i]
        with Image.open(r.resolved_path) as im: im=im.convert("RGB")
        if self.mode == "unlabeled":
            return weak_tf(im), strong_tf(im), i
        if self.mode == "train": return weak_tf(im), int(r.binary_label)
        if self.mode == "train_strong": return strong_tf(im), int(r.binary_label)
        if self.mode == "valcal": return eval_tf(im), int(r.binary_label), i
        return eval_tf(im), int(r.binary_label)

def make_loader(frame, mode, shuffle=False, batch_size=BATCH_SIZE, seed=SEED):
    gen=torch.Generator().manual_seed(seed)
    return DataLoader(CellDataset(frame,mode), batch_size=batch_size, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=False, drop_last=False, generator=gen)

## 3. Backbone, EMA, and metrics

All arms use pretrained ConvNeXt-Tiny with a newly initialized binary head. EMA weights (decay 0.995) are evaluated for every method. The test report includes Macro-F1, AUROC, AUPRC, abnormal recall, specificity, balanced accuracy, and abnormal recall at a validation-selected 90% specificity threshold.

In [4]:
def build_model():
    m=models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.DEFAULT)
    m.classifier[2]=nn.Linear(m.classifier[2].in_features,2)
    return m.to(DEVICE)


def make_ema_model(model):
    ema=copy.deepcopy(model).to(DEVICE)
    ema.eval()
    for p in ema.parameters(): p.requires_grad_(False)
    return ema

@torch.no_grad()
def update_ema(ema, model, decay=EMA_DECAY):
    msd=model.state_dict()
    for k, v in ema.state_dict().items():
        src=msd[k].detach()
        if torch.is_floating_point(v):
            v.mul_(decay).add_(src, alpha=1.0-decay)
        else:
            v.copy_(src)


CLASS_NAMES={0:"Normal",1:"Abnormal"}
def calc_metrics(y, prob):
    y=np.asarray(y,dtype=int); prob=np.asarray(prob,dtype=float); pred=(prob>=0.5).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    try: auc=float(roc_auc_score(y,prob))
    except ValueError: auc=float("nan")
    return {"accuracy":float(accuracy_score(y,pred)), "precision_abnormal":float(precision_score(y,pred,zero_division=0)),
            "abnormal_recall":float(recall_score(y,pred,zero_division=0)),
            "specificity_normal":float(tn/(tn+fp)) if tn+fp else float("nan"),
            "macro_f1":float(f1_score(y,pred,average="macro",zero_division=0)),
            "balanced_accuracy":float(balanced_accuracy_score(y,pred)), "auroc":auc, "auprc":float(average_precision_score(y,prob)) if len(np.unique(y))==2 else float("nan"),
            "tn":int(tn),"fp":int(fp),"fn":int(fn),"tp":int(tp),
            "normal_f1":float(f1_score(y,pred,labels=[0],average="macro",zero_division=0)),
            "abnormal_f1":float(f1_score(y,pred,labels=[1],average="macro",zero_division=0))}

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); ys=[]; probs=[]
    for x,y in loader:
        x=x.to(DEVICE)
        with torch.autocast(device_type=DEVICE.type,enabled=AMP): logits=model(x)
        probs.extend(torch.softmax(logits.float(),dim=1)[:,1].cpu().numpy()); ys.extend(y.numpy())
    return calc_metrics(ys,probs)

class PredictionMemory:
    """Per-image circular buffer of hard classes. Current prediction is written after mask calculation."""
    def __init__(self,n_images,k=MEMORY_K):
        self.values=np.full((n_images,k),-1,dtype=np.int8); self.count=np.zeros(n_images,dtype=np.int16); self.pos=np.zeros(n_images,dtype=np.int16); self.k=k
    def stability(self, ids, current_pred):
        ids=np.asarray(ids,dtype=int); current_pred=np.asarray(current_pred,dtype=int)
        scores=np.zeros(len(ids),dtype=np.float32); counts=self.count[ids].copy()
        for j,(idx,pred) in enumerate(zip(ids,current_pred)):
            n=int(self.count[idx])
            if n: scores[j]=float(np.mean(self.values[idx,:n] == pred))
        return scores, counts
    def write(self, ids, pred):
        for idx,p in zip(np.asarray(ids,dtype=int),np.asarray(pred,dtype=int)):
            slot=int(self.pos[idx]); self.values[idx,slot]=int(p)
            self.pos[idx]=(slot+1)%self.k; self.count[idx]=min(self.k,int(self.count[idx])+1)

def admission_mask(method, conf, weak_pred, strong_pred, stability, history_count):
    conf=np.asarray(conf); weak_pred=np.asarray(weak_pred); strong_pred=np.asarray(strong_pred)
    stability=np.asarray(stability); history_count=np.asarray(history_count)
    high=conf >= TAU_HIGH
    agree=(weak_pred==strong_pred)
    stable=(stability>=STABILITY_MIN)&(history_count>=MEMORY_MIN)
    if method == "fixmatch":
        mask=high
    elif method == "fixmatch_tau80":
        mask=conf >= TAU_LOW_FIXED
    elif method == "cas_match":
        low=(conf>=TAU_LOW_FIXED)&(conf<TAU_HIGH)&agree&stable
        mask=high|low
    else:
        raise ValueError(f"Method has no pseudo-label admission rule: {method}")
    band=np.where(conf>=TAU_HIGH,"high",np.where(conf>=TAU_LOW_FIXED,"low","below_low"))
    route=np.where(mask,band,"rejected")
    return mask, route, agree, stable, band


## 4. Fixed pseudo-label thresholds

No validation-tuned τ_low is used. FixMatch-0.95 admits confidence ≥0.95. FixMatch-0.80 admits confidence ≥0.80 with no gates. CAS-Match admits confidence ≥0.95 automatically and admits the 0.80–0.95 band only when weak/strong agreement AND temporal stability pass. This makes the CAS-Match comparison explicit: the gates are tested against a no-gate τ=0.80 control.

In [5]:
print(f"Locked thresholds: high={TAU_HIGH:.2f}, low={TAU_LOW_FIXED:.2f}")
print("CAS-Match admits the 0.80–0.95 band only when weak/strong predictions agree AND temporal stability passes.")
print("FixMatch-0.80 admits all predictions >=0.80 without gates; FixMatch-0.95 admits >=0.95.")
print("No validation-fitted threshold or silent fallback is used in this protocol.")


Locked thresholds: high=0.95, low=0.80
CAS-Match admits the 0.80–0.95 band only when weak/strong predictions agree AND temporal stability passes.
FixMatch-0.80 admits all predictions >=0.80 without gates; FixMatch-0.95 admits >=0.95.
No validation-fitted threshold or silent fallback is used in this protocol.


## 5. Checkpointing and reproducibility

Every run has a run-specific hash that includes its method, seed, budget, label-subset hash, validation/test hashes, and protocol settings. Latest checkpoints contain student and EMA weights, optimizer/scaler state, temporal memory, histories, and RNG states. Writes are atomic; the prior completed checkpoint is retained for recovery.

In [9]:
def stable_seed(text): return int(hashlib.sha256(text.encode()).hexdigest()[:8],16)
def run_name(ds,frac,method,seed): return f"{ds}_{int(round(frac*100))}pct_{method}_seed{seed}"

def make_unlabeled_epoch_loader(frame, epoch, run_id, seed):
    gen=torch.Generator().manual_seed(seed+epoch)
    ds=CellDataset(frame,"unlabeled")
    return DataLoader(ds,batch_size=BATCH_SIZE,shuffle=True,num_workers=0,
                      pin_memory=False,drop_last=False,generator=gen)


def aggregate_epoch_diagnostics(rows, epoch):
    """Collapse per-batch audit rows into 24 epoch-level strata."""
    keys=["predicted_class","confidence_band","agreement","stable"]
    count_cols=[c for c in (
        "candidate_n","candidate_correct_n","candidate_true_normal_n","candidate_true_abnormal_n",
        "accepted_n","accepted_correct_n","accepted_true_normal_n","accepted_true_abnormal_n",
        "rejected_n","rejected_correct_n","rejected_true_normal_n","rejected_true_abnormal_n") ]
    if not rows:
        return pd.DataFrame(columns=["epoch"]+keys+count_cols+["candidate_precision","accepted_precision","rejected_precision"])
    frame=pd.DataFrame(rows)
    out=frame.groupby(keys,as_index=False,dropna=False)[count_cols].sum()
    for prefix in ("candidate","accepted","rejected"):
        out[prefix+"_precision"]=out[prefix+"_correct_n"]/out[prefix+"_n"].replace(0,np.nan)
    out.insert(0,"epoch",int(epoch))
    return out

def append_epoch_diagnostics(run_dir, rows, epoch, last_completed_epoch=None):
    """Append one aggregated epoch; on resume discard any uncommitted future rows."""
    path=Path(run_dir)/"pseudo_label_epoch_diagnostics.csv"
    if path.exists():
        old=pd.read_csv(path)
        if "epoch" not in old.columns:
            # A pre-fix per-batch file cannot be safely mapped to epochs. Preserve it separately.
            legacy=path.with_name("pseudo_label_epoch_diagnostics_legacy_unaggregated.csv")
            if not legacy.exists(): old.to_csv(legacy,index=False)
            old=pd.DataFrame()
        if last_completed_epoch is not None and len(old):
            old=old[old.epoch.astype(int)<=int(last_completed_epoch)]
    else:
        old=pd.DataFrame()
    current=aggregate_epoch_diagnostics(rows,epoch)
    combined=pd.concat([old,current],ignore_index=True)
    combined.to_csv(path,index=False)
    return path

def atomic_torch_save(state, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".writing")

    # Write, flush and sync using a writable file handle.
    with open(tmp, "wb") as f:
        torch.save(state, f)
        f.flush()
        os.fsync(f.fileno())

    if path.exists():
        previous = path.with_name(path.name + ".previous")
        if previous.exists():
            previous.unlink()
        os.replace(path, previous)

    os.replace(tmp, path)

def save_checkpoint(path, model, ema_model, optimizer, scaler, epoch, best_score, bad_epochs,
                    history, train_memory, seed, config_hash):
    state={"model":model.state_dict(),"ema_model":ema_model.state_dict(),
           "optimizer":optimizer.state_dict(),"scaler":scaler.state_dict(),
           "epoch":epoch,"best_score":best_score,"bad_epochs":bad_epochs,"history":history,
           "train_memory":train_memory.__dict__,
           "seed":seed,"python_rng":random.getstate(),"numpy_rng":np.random.get_state(),
           "torch_rng":torch.get_rng_state(),
           "cuda_rng":torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
           "config_hash":config_hash}
    atomic_torch_save(state,path)

def restore_memory(obj,state):
    obj.values=state["values"]; obj.count=state["count"]; obj.pos=state["pos"]; obj.k=state["k"]

def load_checkpoint(path,model,ema_model,optimizer,scaler,train_memory,expected_hash,seed):
    st=torch.load(path,map_location=DEVICE,weights_only=False)
    if st.get("config_hash")!=expected_hash: raise ValueError("Checkpoint config hash mismatch.")
    if int(st.get("seed",-1))!=int(seed): raise ValueError("Checkpoint seed mismatch.")
    model.load_state_dict(st["model"]); ema_model.load_state_dict(st["ema_model"])
    optimizer.load_state_dict(st["optimizer"]); scaler.load_state_dict(st["scaler"])
    restore_memory(train_memory,st["train_memory"])
    random.setstate(st["python_rng"]); np.random.set_state(st["numpy_rng"])
    torch.set_rng_state(st["torch_rng"].cpu())
    if torch.cuda.is_available() and st.get("cuda_rng") is not None:
        cr=st["cuda_rng"]; cr=[cr] if isinstance(cr,torch.Tensor) else cr
        torch.cuda.set_rng_state_all([torch.as_tensor(x,dtype=torch.uint8,device="cpu").contiguous() for x in cr])
    return st


## 6. Training protocol

All arms use the same EMA evaluation and smoothed validation Macro-F1 selection rule. Low-label supervised arms and their paired SSL arms use exactly the same number of optimizer steps per epoch: the number of unlabeled minibatches in the corresponding SSL run. They also use the same seed-determined labeled-example sampling sequence, with replacement, so small labeled subsets are cycled without giving supervised fewer updates. The 100%-label reference uses one ordinary shuffled pass through the full labeled training set per epoch. No checkpoint is eligible before epoch 8. Test data is not used for model selection.


In [10]:
def predict_probs(model, frame):
    model.eval()
    loader=make_loader(frame,"eval",shuffle=False,batch_size=BATCH_SIZE)
    ys=[]; probs=[]
    with torch.no_grad():
        for x,y in loader:
            x=x.to(DEVICE)
            with torch.autocast(device_type=DEVICE.type,enabled=AMP):
                logits=model(x)
            probs.extend(torch.softmax(logits.float(),dim=1)[:,1].cpu().numpy())
            ys.extend(y.numpy())
    return np.asarray(ys,dtype=int),np.asarray(probs,dtype=float)

def recall_at_specificity(y, prob, target=TARGET_SPECIFICITY):
    y=np.asarray(y,dtype=int); prob=np.asarray(prob,dtype=float)
    normal_scores=prob[y==0]
    if len(normal_scores)==0: return float("nan"),float("nan")
    threshold=float(np.quantile(normal_scores,target,method="higher"))
    pred=(prob>=threshold).astype(int)
    specificity=float(np.mean(pred[y==0]==0))
    abnormal_recall=float(np.mean(pred[y==1]==1)) if np.any(y==1) else float("nan")
    return abnormal_recall,threshold

def grouped_route_diagnostics(pred, conf, strong_pred, stability, history_count,
                              mask, y_true):
    rows=[]
    agree=(pred==strong_pred)
    stable=(stability>=STABILITY_MIN)&(history_count>=MEMORY_MIN)
    band=np.where(conf>=TAU_HIGH,"high",
                  np.where(conf>=TAU_LOW_FIXED,"low","below_low"))
    for cls in (0,1):
        for b in ("high","low","below_low"):
            for ag in (False,True):
                for st in (False,True):
                    candidate=(pred==cls)&(band==b)&(agree==ag)&(stable==st)
                    admitted=candidate&mask
                    rejected=candidate&(~mask)
                    n=int(candidate.sum()); an=int(admitted.sum()); rn=int(rejected.sum())
                    candidate_correct=int((candidate&(pred==y_true)).sum())
                    accepted_correct=int((admitted&(pred==y_true)).sum())
                    rejected_correct=int((rejected&(pred==y_true)).sum())
                    rows.append({
                      "predicted_class":cls,"confidence_band":b,
                      "agreement":ag,"stable":st,
                      "candidate_n":n,"candidate_correct_n":candidate_correct,
                      "candidate_precision":(candidate_correct/n if n else np.nan),
                      "candidate_true_normal_n":int((candidate&(y_true==0)).sum()),
                      "candidate_true_abnormal_n":int((candidate&(y_true==1)).sum()),
                      "accepted_n":an,"accepted_correct_n":accepted_correct,
                      "accepted_precision":(accepted_correct/an if an else np.nan),
                      "accepted_true_normal_n":int((admitted&(y_true==0)).sum()),
                      "accepted_true_abnormal_n":int((admitted&(y_true==1)).sum()),
                      "rejected_n":rn,"rejected_correct_n":rejected_correct,
                      "rejected_precision":(rejected_correct/rn if rn else np.nan),
                      "rejected_true_normal_n":int((rejected&(y_true==0)).sum()),
                      "rejected_true_abnormal_n":int((rejected&(y_true==1)).sum()),
                    })
    return rows

def train_experiment(ds,frac,method,seed,max_epochs=None):
    max_epochs=MAX_EPOCHS if max_epochs is None else int(max_epochs)
    run_id=run_name(ds,frac,method,seed)
    rd=OUT_DIR/run_id; rd.mkdir(parents=True,exist_ok=True)
    latest=rd/"checkpoint_latest.pt"; best=rd/"checkpoint_best.pt"; done=rd/"DONE.json"
    lab,unlab=labeled_frames(ds,frac,seed)
    val_df=manifests[ds].query("split == 'val'").reset_index(drop=True)
    test_df=manifests[ds].query("split == 'test'").reset_index(drop=True)
    audit=protocol_split_audit(ds,frac,seed,[method])
    run_config={**CONFIG,"dataset":ds,"fraction":frac,"method":method,"seed":seed,
                "labeled_subset_sha256":audit["labeled_subset_sha256"],
                "validation_sha256":audit["validation_sha256"],
                "test_sha256":audit["test_sha256"]}
    run_hash=hashlib.sha256(json.dumps(run_config,sort_keys=True).encode()).hexdigest()[:20]
    protocol_path=rd/"protocol.json"
    if protocol_path.exists():
        stored=json.loads(protocol_path.read_text())
        if stored.get("run_hash") != run_hash:
            raise RuntimeError(f"Stored protocol hash mismatch for {run_id}: "+
                               f"stored={stored.get('run_hash')} current={run_hash}. Use a fresh v3 output folder; do not reuse this run directory.")
    if done.exists() and (rd/"test_metrics.json").exists():
        stored_done=json.loads(done.read_text())
        metrics=json.loads((rd/"test_metrics.json").read_text())
        if stored_done.get("run_hash") != run_hash or metrics.get("run_hash") != run_hash:
            raise RuntimeError(f"Completed run protocol hash mismatch for {run_id}; refusing to skip it.")
        for name in ("checkpoint_latest.pt","checkpoint_latest.pt.previous",
                     "checkpoint_best.pt","checkpoint_best.pt.previous",
                     "checkpoint_latest.pt.writing","checkpoint_best.pt.writing"):
            q=rd/name
            if q.exists(): q.unlink()
        print("Already completed with matching protocol:",run_id)
        return metrics
    if done.exists(): raise RuntimeError(f"Incomplete completion marker in {rd}")
    protocol_path.write_text(json.dumps({**run_config,"run_hash":run_hash},indent=2))
    assert_disk_headroom()
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    seed_all(seed)
    model=build_model()
    ema_model=make_ema_model(model)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    scaler=torch.amp.GradScaler("cuda",enabled=AMP)
    train_memory=PredictionMemory(len(unlab),MEMORY_K)
    history=[]; best_score=-float("inf"); bad_epochs=0; start_epoch=0

    backup=latest.with_name(latest.name+".previous")
    if latest.exists() or backup.exists():
        primary=latest if latest.exists() else backup
        try:
            st=load_checkpoint(primary,model,ema_model,optimizer,scaler,
                               train_memory,run_hash,seed)
        except Exception as e:
            if primary == backup or not backup.exists():
                raise RuntimeError(f"Latest and/or backup checkpoint could not be loaded for {run_id}") from e
            st=load_checkpoint(backup,model,ema_model,optimizer,scaler,
                               train_memory,run_hash,seed)
            print("Recovered from previous completed-epoch checkpoint.")
        if primary == backup:
            shutil.copy2(backup,latest)
        start_epoch=int(st["epoch"])+1
        best_score=float(st["best_score"]); bad_epochs=int(st["bad_epochs"])
        history=st["history"]
        last_completed_epoch=int(st["epoch"])+1
        diag_path=rd/"pseudo_label_epoch_diagnostics.csv"
        if diag_path.exists():
            saved_diag=pd.read_csv(diag_path)
            if "epoch" in saved_diag.columns:
                saved_diag=saved_diag[saved_diag.epoch.astype(int)<=last_completed_epoch]
                saved_diag.to_csv(diag_path,index=False)
            else:
                legacy=diag_path.with_name("pseudo_label_epoch_diagnostics_legacy_unaggregated.csv")
                if not legacy.exists(): saved_diag.to_csv(legacy,index=False)
                diag_path.unlink()
        print(f"Resuming {run_id} at epoch {start_epoch+1}")

    for epoch in range(start_epoch,max_epochs):
        assert_disk_headroom()
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        model.train(); epoch_losses=[]; diag=[]
        if method in ("supervised","supervised_strong","supervised_full"):
            mode="train_strong" if method=="supervised_strong" else "train"
            train_ds=CellDataset(lab,mode)
            if method=="supervised_full":
                # Full-label reference: one ordinary shuffled pass over all labeled training data.
                lab_loader=DataLoader(train_ds,batch_size=BATCH_SIZE,shuffle=True,
                                      num_workers=0,pin_memory=False,drop_last=False,
                                      generator=torch.Generator().manual_seed(seed+epoch+777))
                for x,y in lab_loader:
                    x=x.to(DEVICE); y=y.to(DEVICE)
                    optimizer.zero_grad(set_to_none=True)
                    with torch.autocast(device_type=DEVICE.type,enabled=AMP):
                        logits=model(x); loss=F.cross_entropy(logits,y)
                    scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
                    update_ema(ema_model,model)
                    epoch_losses.append(float(loss.detach().cpu()))
                del lab_loader
            else:
                # Same optimizer-step count and same sampled labeled indices as paired SSL.
                steps_per_epoch=math.ceil(len(unlab)/BATCH_SIZE)
                rng=np.random.default_rng(seed+epoch+777)
                for step in range(steps_per_epoch):
                    idxs=rng.choice(len(train_ds),size=min(BATCH_SIZE,len(train_ds)),replace=True)
                    batch=[train_ds[int(j)] for j in idxs]
                    x=torch.stack([z[0] for z in batch]).to(DEVICE)
                    y=torch.tensor([z[1] for z in batch],dtype=torch.long,device=DEVICE)
                    optimizer.zero_grad(set_to_none=True)
                    with torch.autocast(device_type=DEVICE.type,enabled=AMP):
                        logits=model(x); loss=F.cross_entropy(logits,y)
                    scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
                    update_ema(ema_model,model)
                    epoch_losses.append(float(loss.detach().cpu()))
        else:
            # Match the labeled minibatch sampling sequence across supervised and SSL arms.
            rng=np.random.default_rng(seed+epoch+777)
            lab_ds=CellDataset(lab,"train")
            unl_loader=make_unlabeled_epoch_loader(unlab,epoch,run_id,seed)
            for bi,(uw,us,uids) in enumerate(unl_loader):
                idxs=rng.choice(len(lab_ds),size=min(BATCH_SIZE,len(lab_ds)),replace=True)
                batch=[lab_ds[int(j)] for j in idxs]
                x=torch.stack([z[0] for z in batch]).to(DEVICE)
                y=torch.tensor([z[1] for z in batch],dtype=torch.long,device=DEVICE)
                uw=uw.to(DEVICE); us=us.to(DEVICE); ids=uids.numpy().astype(int)
                optimizer.zero_grad(set_to_none=True)
                with torch.autocast(device_type=DEVICE.type,enabled=AMP):
                    logits_l=model(x); loss_s=F.cross_entropy(logits_l,y)
                    model.eval()  # deterministic pseudo-label views; disables stochastic depth
                    with torch.no_grad():
                        pw=torch.softmax(model(uw).float(),dim=1)
                        conf,pseudo=pw.max(dim=1)
                        ps_pred=model(us).float().argmax(dim=1)
                        stab,hcount=train_memory.stability(ids,pseudo.cpu().numpy())
                        mask_np,route,agree,stable,band=admission_mask(
                            method,conf.cpu().numpy(),pseudo.cpu().numpy(),ps_pred.cpu().numpy(),
                            stab,hcount)
                        mask=torch.as_tensor(mask_np,dtype=torch.bool,device=DEVICE)
                    model.train()
                    logits_u=model(us)
                    per_sample=F.cross_entropy(logits_u,pseudo.detach(),reduction="none")
                    loss_u=(per_sample*mask.float()).mean()
                    loss=loss_s+UNSUP_WEIGHT*loss_u
                scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
                update_ema(ema_model,model)
                train_memory.write(ids,pseudo.cpu().numpy())
                epoch_losses.append(float(loss.detach().cpu()))
                # Ground truth is used only after the admission decision and gradient update.
                y_audit=unlab.iloc[ids].binary_label.to_numpy(dtype=int)
                diag.extend(grouped_route_diagnostics(
                    pseudo.cpu().numpy(),conf.cpu().numpy(),ps_pred.cpu().numpy(),
                    stab,hcount,mask_np,y_audit))
            del unl_loader

        val_y,val_prob=predict_probs(ema_model,val_df)
        val_metrics=calc_metrics(val_y,val_prob)
        smooth_values=[float(h["val_macro_f1_ema"]) for h in history[-(VAL_SMOOTH_WINDOW-1):]]
        smooth_values.append(float(val_metrics["macro_f1"]))
        smooth_score=float(np.mean(smooth_values))
        row={"epoch":epoch+1,"optimizer_steps":len(epoch_losses),
             "train_loss":float(np.mean(epoch_losses)) if epoch_losses else np.nan,
             "val_macro_f1_ema":val_metrics["macro_f1"],"val_auroc_ema":val_metrics["auroc"],
             "val_auprc_ema":val_metrics["auprc"],"val_macro_f1_smoothed":smooth_score,
             "pseudo_accepted_n":int(sum(x["accepted_n"] for x in diag)),
             "pseudo_pool_n":int(sum(x["candidate_n"] for x in diag)),
             "pseudo_acceptance_rate":(sum(x["accepted_n"] for x in diag)/max(sum(x["candidate_n"] for x in diag),1)) if diag else 0.0,
             "tau_high":TAU_HIGH,"tau_low_fixed":TAU_LOW_FIXED}
        history.append(row)
        pd.DataFrame(history).to_csv(rd/"history.csv",index=False)
        append_epoch_diagnostics(rd,diag,epoch+1,last_completed_epoch=None)

        # Same EMA + 3-epoch smoothed validation rule for every method.
        eligible=(epoch+1)>=MIN_SELECT_EPOCH
        if eligible and smooth_score>best_score:
            best_score=smooth_score; bad_epochs=0
            atomic_torch_save({"ema_model":ema_model.state_dict(),"epoch":epoch+1,
                        "val_metrics":val_metrics,"smoothed_val_macro_f1":smooth_score,
                        "config_hash":run_hash},best)
        elif eligible:
            bad_epochs+=1
        save_checkpoint(latest,model,ema_model,optimizer,scaler,epoch,best_score,bad_epochs,
                        history,train_memory,seed,run_hash)
        print(f"{run_id} | epoch {epoch+1}/{max_epochs} | EMA val F1={val_metrics['macro_f1']:.4f} | smooth={smooth_score:.4f}")
        del epoch_losses,diag
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        if eligible and bad_epochs>=PATIENCE:
            print("Early stopping after minimum selection epoch."); break

    if not best.exists(): raise RuntimeError(f"No eligible best checkpoint for {run_id}")
    try:
        best_payload=torch.load(best,map_location=DEVICE,weights_only=False)
    except Exception:
        best_backup=best.with_name(best.name+".previous")
        if not best_backup.exists(): raise
        best_payload=torch.load(best_backup,map_location=DEVICE,weights_only=False)
        shutil.copy2(best_backup,best)
        print("Recovered best EMA checkpoint from previous copy.")
    if best_payload.get("config_hash") != run_hash:
        raise RuntimeError("Best checkpoint protocol hash mismatch.")
    ema_model.load_state_dict(best_payload["ema_model"])
    # Validation-derived threshold for 90% specificity; test set is not used to select it.
    vy,vp=predict_probs(ema_model,val_df)
    val_recall90,val_threshold90=recall_at_specificity(vy,vp,TARGET_SPECIFICITY)
    ty,tp=predict_probs(ema_model,test_df)
    test_metrics=calc_metrics(ty,tp)
    test_pred90=(tp>=val_threshold90).astype(int)
    test_specificity90=float(np.mean(test_pred90[ty==0]==0))
    test_abnormal_recall90=float(np.mean(test_pred90[ty==1]==1))
    test_metrics.update({"abnormal_recall_at_val_90pct_specificity":test_abnormal_recall90,
                         "specificity_at_val_90pct_threshold":test_specificity90,
                         "val_threshold_for_90pct_specificity":val_threshold90,
                         "val_abnormal_recall_at_90pct_specificity":val_recall90})
    prob_df=pd.DataFrame({"resolved_path":test_df.resolved_path.astype(str),
                          "true_label":ty,"p_abnormal":tp,"predicted_at_0_5":(tp>=0.5).astype(int),
                          "predicted_at_val_90pct_specificity":test_pred90})
    prob_df.to_csv(rd/"test_probabilities.csv",index=False)
    result={"run_id":run_id,"dataset":ds,"method":method,"label_fraction":frac,"seed":seed,
            "best_epoch":best_payload["epoch"],
            "best_val_macro_f1_smoothed":best_payload["smoothed_val_macro_f1"],
            "test_metrics":test_metrics,"run_hash":run_hash,
            "test_probabilities":"test_probabilities.csv"}
    (rd/"test_metrics.json").write_text(json.dumps(result,indent=2))
    done.write_text(json.dumps({"completed":True,"run_id":run_id,"run_hash":run_hash},indent=2))
    # Checkpoints are no longer needed once test metrics and probabilities are durable.
    for name in ("checkpoint_latest.pt","checkpoint_latest.pt.previous",
                 "checkpoint_best.pt","checkpoint_best.pt.previous",
                 "checkpoint_latest.pt.writing","checkpoint_best.pt.writing"):
        q=rd/name
        if q.exists():
            try: q.unlink()
            except OSError as e: print("WARNING: could not remove completed checkpoint",q,e)
    del model,ema_model
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return result


## 7. Smoke tests before training

Run these synthetic checks first. They validate fixed threshold admission, the temporal warm-up, four-way agreement/stability diagnostics, and atomic checkpoint writes. They do not require a GPU or training data.

In [11]:
# Protocol smoke tests: no training, no dataset access required.
mem=PredictionMemory(n_images=2,k=MEMORY_K)
ids=np.array([0]); current=np.array([1])
stab,count=mem.stability(ids,current)
assert count[0]==0 and stab[0]==0
for _ in range(MEMORY_MIN): mem.write(ids,current)
stab,count=mem.stability(ids,current)
assert count[0]==MEMORY_MIN and abs(stab[0]-1.0)<1e-6
conf=np.array([0.99,0.90,0.90,0.90,0.75])
wp=np.array([1,1,1,1,1]); sp=np.array([0,1,1,0,1])
st=np.array([0.0,1.0,0.6,1.0,1.0]); hc=np.array([0,3,3,3,3])
m95,r95,ag,stable,band=admission_mask("fixmatch",conf,wp,sp,st,hc)
m80,r80,_,_,_=admission_mask("fixmatch_tau80",conf,wp,sp,st,hc)
mcas,rcas,_,_,_=admission_mask("cas_match",conf,wp,sp,st,hc)
assert m95.tolist()==[True,False,False,False,False]
assert m80.tolist()==[True,True,True,True,False]
assert mcas.tolist()==[True,True,False,False,False]
assert rcas[1]=="low" and rcas[2]=="rejected"
y=np.array([1,1,0,1,0])
rows=grouped_route_diagnostics(wp,conf,sp,st,hc,mcas,y)
assert len(rows)==24
assert all("candidate_correct_n" in r and "accepted_correct_n" in r and "rejected_correct_n" in r for r in rows)
assert all("candidate_true_normal_n" in r and "candidate_true_abnormal_n" in r for r in rows)
assert sum(r["candidate_n"] for r in rows)==len(y), "Every pool item must be counted exactly once."
accepted=sum(r["accepted_n"] for r in rows); pool=sum(r["candidate_n"] for r in rows)
assert accepted==int(mcas.sum()) and pool==len(y)
assert abs(accepted/pool - 0.4)<1e-9, "Acceptance rate must use the full pool denominator."
from tempfile import TemporaryDirectory
with TemporaryDirectory() as td:
    q=Path(td)/"smoke.pt"; atomic_torch_save({"ok":True},q)
    assert torch.load(q,weights_only=False)["ok"] is True
print("PASS: fixed thresholds, gate logic, full-pool denominator, candidate/accepted/rejected correctness, atomic checkpoint writes.")


PASS: fixed thresholds, gate logic, full-pool denominator, candidate/accepted/rejected correctness, atomic checkpoint writes.


## 8. Timing preflight and run queue

First run the optional benchmark cell immediately below: `benchmark_ssl_epoch("cric", 0.05, 42)`. It trains one FixMatch-style epoch in memory, writes no checkpoints or result files, and reports a lower-bound elapsed time and rough full-run estimate. It omits EMA updates, validation, and some memory/diagnostics bookkeeping, so actual epochs will take longer. If the runtime is unacceptable, adjust the compute plan before launching the queue. Then run the seed-by-seed queue cell. The queue contains 41 runs and skips completed runs on restart.


In [12]:
def benchmark_ssl_epoch(ds="cric",frac=0.05,seed=42):
    """Time one FixMatch-style training epoch; estimate is a lower bound (omits EMA, validation and full diagnostics bookkeeping)."""
    assert_disk_headroom()
    lab,unlab=labeled_frames(ds,frac,seed)
    seed_all(seed)
    model=build_model()
    optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY)
    scaler=torch.amp.GradScaler("cuda",enabled=AMP)
    lab_ds=CellDataset(lab,"train")
    loader=make_unlabeled_epoch_loader(unlab,0,f"benchmark_{ds}_{frac}_{seed}",seed)
    rng=np.random.default_rng(seed+777)
    losses=[]; t0=time.perf_counter()
    for uw,us,uids in loader:
        idxs=rng.choice(len(lab_ds),size=min(BATCH_SIZE,len(lab_ds)),replace=True)
        batch=[lab_ds[int(j)] for j in idxs]
        x=torch.stack([z[0] for z in batch]).to(DEVICE)
        y=torch.tensor([z[1] for z in batch],dtype=torch.long,device=DEVICE)
        uw=uw.to(DEVICE); us=us.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=DEVICE.type,enabled=AMP):
            logits_l=model(x); loss_s=F.cross_entropy(logits_l,y)
            model.eval()
            with torch.no_grad():
                pw=torch.softmax(model(uw).float(),dim=1)
                conf,pseudo=pw.max(dim=1)
            model.train()
            logits_u=model(us)
            mask=(conf>=TAU_HIGH).float()
            loss_u=(F.cross_entropy(logits_u,pseudo,reduction="none")*mask).mean()
            loss=loss_s+UNSUP_WEIGHT*loss_u
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        losses.append(float(loss.detach().cpu()))
    elapsed=time.perf_counter()-t0
    steps=len(losses)
    print(f"Benchmark: {steps} optimizer steps in {elapsed/60:.1f} min "
          f"({elapsed/max(steps,1):.2f} sec/step).")
    print(f"Lower-bound epoch estimate (excludes EMA/validation/full diagnostics): {elapsed/60:.1f} min; 60-epoch lower-bound estimate: {elapsed*60/3600:.1f} hours.")
    del model,optimizer,scaler,loader,lab_ds
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return {"steps":steps,"seconds":elapsed,"seconds_per_step":elapsed/max(steps,1)}

print("Optional preflight: run benchmark_ssl_epoch('cric', 0.05, 42) to time one complete FixMatch epoch.")


Optional preflight: run benchmark_ssl_epoch('cric', 0.05, 42) to time one complete FixMatch epoch.


In [13]:
# SEED-BY-SEED PRE-REGISTERED QUEUE. Safe to interrupt and rerun.
# Each seed 42-44 runs every arm/budget before the next seed begins.
# Seeds 45-46 extend only the primary FixMatch(0.95) vs CAS-Match comparison.
RUN_SELECTION = []
CORE_ARMS = [
    ("supervised", (0.05,0.10)),
    ("supervised_strong", (0.05,0.10)),
    ("supervised_full", (1.0,)),
    ("fixmatch", (0.05,0.10)),
    ("fixmatch_tau80", (0.05,0.10)),
    ("cas_match", (0.05,0.10)),
]
for seed in SEEDS_CORE:
    for method,fracs in CORE_ARMS:
        for frac in fracs:
            RUN_SELECTION.append(("cric",frac,method,seed))
for seed in (45,46):
    for method in ("fixmatch","cas_match"):
        for frac in (0.05,0.10):
            RUN_SELECTION.append(("cric",frac,method,seed))

print("Planned runs:",len(RUN_SELECTION))
print("First seed block:",[x for x in RUN_SELECTION if x[3]==42])
resource_report()
all_results=[]
summary_path=OUT_DIR/"cric_final_protocol_summary.csv"
for ds,frac,method,seed in RUN_SELECTION:
    print("\\n"+"="*76+"\\n"+run_name(ds,frac,method,seed))
    result=train_experiment(ds,frac,method,seed)
    all_results.append(result)
    flat={k:v for k,v in result.items() if k!="test_metrics"}
    flat.update(result["test_metrics"])
    old=pd.read_csv(summary_path) if summary_path.exists() else pd.DataFrame()
    newrow=pd.DataFrame([flat])
    if len(old) and "run_id" in old.columns:
        old=old[old.run_id!=result["run_id"]]
    pd.concat([old,newrow],ignore_index=True).to_csv(summary_path,index=False)
    resource_report()
print("Priority experiment queue completed or paused. Summary:",summary_path)


Planned runs: 41
First seed block: [('cric', 0.05, 'supervised', 42), ('cric', 0.1, 'supervised', 42), ('cric', 0.05, 'supervised_strong', 42), ('cric', 0.1, 'supervised_strong', 42), ('cric', 1.0, 'supervised_full', 42), ('cric', 0.05, 'fixmatch', 42), ('cric', 0.1, 'fixmatch', 42), ('cric', 0.05, 'fixmatch_tau80', 42), ('cric', 0.1, 'fixmatch_tau80', 42), ('cric', 0.05, 'cas_match', 42), ('cric', 0.1, 'cas_match', 42)]
RAM: available=1.22 GB / total=15.23 GB
Disk at output: free=8.05 GB / total=200.12 GB
CUDA allocated=0.00 GB; reserved=0.00 GB
\n============================================================================\ncric_5pct_supervised_seed42
cric_5pct_supervised_seed42 | epoch 1/60 | EMA val F1=0.8279 | smooth=0.8279


RuntimeError: STOPPED SAFELY: only 7.64 GB free; need 8.0 GB. Free space before running.

In [19]:
import torch

p = OUT_DIR / "cric_5pct_supervised_seed42" / "checkpoint_latest.pt"
ckpt = torch.load(p, map_location="cpu", weights_only=False)

def tensor_size_mb(obj):
    if torch.is_tensor(obj):
        return obj.numel() * obj.element_size() / 1024**2
    if isinstance(obj, dict):
        return sum(tensor_size_mb(v) for v in obj.values())
    if isinstance(obj, (list, tuple)):
        return sum(tensor_size_mb(v) for v in obj)
    return 0

for key, value in ckpt.items():
    print(f"{key:15s}: {tensor_size_mb(value):8.2f} MB")

model          :   106.13 MB
ema_model      :   106.13 MB
optimizer      :   212.26 MB
scaler         :     0.00 MB
epoch          :     0.00 MB
best_score     :     0.00 MB
bad_epochs     :     0.00 MB
history        :     0.00 MB
train_memory   :     0.00 MB
seed           :     0.00 MB
python_rng     :     0.00 MB
numpy_rng      :     0.00 MB
torch_rng      :     0.00 MB
cuda_rng       :     0.00 MB
config_hash    :     0.00 MB


## 9. Stop, resume, and storage

Interrupt the kernel at any time. An interrupted epoch may replay, but completed epochs are checkpointed. After restarting the kernel, rerun setup cells in order, pass the smoke tests, optionally rerun the timing benchmark, then rerun the queue cell. Keep at least 8 GB free. After a run has saved its test metrics, test probabilities and `DONE.json`, its bulky checkpoints are automatically deleted. Completed runs are still recognized and skipped because their metrics and completion marker remain. Incomplete runs retain latest and previous checkpoints for recovery.


## 10. Results and interpretation

The summary cell reports per-run metrics and paired deltas for CAS-Match vs FixMatch-0.95, CAS-Match vs FixMatch-0.80, FixMatch vs standard supervised, CAS-Match vs standard supervised, and strong-augmentation supervised vs standard supervised on common seeds 42–44. It also reports the primary CAS-Match vs FixMatch-0.95 deltas on seeds 42–46. Apply the pre-registered 1-point Macro-F1 and seed-consistency rules in the introduction. Report means, standard deviations and per-seed deltas. Test-image bootstraps are supplementary and conditional on each trained model.


In [ ]:
summary_path=OUT_DIR/"cric_final_protocol_summary.csv"
if summary_path.exists():
    summary=pd.read_csv(summary_path)
    display(summary.sort_values(["seed","label_fraction","method"]))
    metric_cols=["macro_f1","auroc","auprc","abnormal_recall",
                 "specificity_normal","balanced_accuracy",
                 "abnormal_recall_at_val_90pct_specificity"]
    available=[m for m in metric_cols if m in summary.columns]
    # Main descriptive summaries. Also show common-seed means for fair arm comparisons.
    display(summary.groupby(["method","label_fraction"])[available].agg(["mean","std","count"]))
    common=summary[summary.seed.isin(SEEDS_CORE)].copy()
    print("Arm means restricted to common seeds 42-44 (three-seed comparisons):")
    display(common.groupby(["method","label_fraction"])[available].agg(["mean","std","count"]))
    common=common[common.method!="supervised_full"]
    pairs=[
      ("cas_match","fixmatch","CAS-Match minus FixMatch-0.95"),
      ("cas_match","fixmatch_tau80","CAS-Match minus FixMatch-0.80"),
      ("fixmatch_tau80","fixmatch","FixMatch-0.80 minus FixMatch-0.95"),
      ("fixmatch","supervised","FixMatch minus supervised"),
      ("cas_match","supervised","CAS-Match minus supervised"),
      ("supervised_strong","supervised","Strong-augmentation supervised minus standard supervised"),
    ]
    paired_rows=[]
    for frac in (0.05,0.10):
        for left,right,label in pairs:
            l=common[(common.label_fraction==frac)&(common.method==left)][["seed"]+available]
            r=common[(common.label_fraction==frac)&(common.method==right)][["seed"]+available]
            joined=l.merge(r,on="seed",suffixes=("_left","_right"))
            for _,row in joined.iterrows():
                rec={"label_fraction":frac,"comparison":label,"seed":int(row["seed"])}
                for metric in available:
                    rec[metric+"_delta_left_minus_right"]=float(row[metric+"_left"]-row[metric+"_right"])
                paired_rows.append(rec)
    paired=pd.DataFrame(paired_rows)
    display(paired.sort_values(["label_fraction","comparison","seed"]))
    if len(paired):
        delta_cols=[m+"_delta_left_minus_right" for m in available]
        display(paired.groupby(["label_fraction","comparison"])[delta_cols].agg(["mean","std","count"]))
    def classify_macro_f1_delta(deltas):
        vals=np.asarray(deltas,dtype=float)
        vals=vals[np.isfinite(vals)]
        if len(vals)==0: return "insufficient paired runs"
        mean_pp=float(vals.mean()*100.0)
        positive=int((vals>0).sum()); negative=int((vals<0).sum()); n=len(vals)
        required=(4 if n>=5 else n)
        if mean_pp>=1.0 and positive>=required: return "win"
        if mean_pp<=-1.0 and negative>=required: return "loss"
        if abs(mean_pp)<1.0: return "approximately equivalent (≈)"
        return "inconclusive"
    print("Pre-registered interpretation for three-seed control comparisons:")
    for (frac,label),group in paired.groupby(["label_fraction","comparison"]):
        col="macro_f1_delta_left_minus_right"
        if col in group:
            print(f"{frac:.0%} | {label}: mean delta={group[col].mean()*100:.2f} pp; "
                  f"seed deltas={(group[col]*100).round(2).tolist()} | "
                  f"{classify_macro_f1_delta(group[col])}")
    # Primary comparison on all five common FixMatch/CAS-Match seeds.
    primary=summary[summary.method.isin(["fixmatch","cas_match"])]
    primary_rows=[]
    for frac in (0.05,0.10):
        l=primary[(primary.label_fraction==frac)&(primary.method=="cas_match")][["seed"]+available]
        r=primary[(primary.label_fraction==frac)&(primary.method=="fixmatch")][["seed"]+available]
        joined=l.merge(r,on="seed",suffixes=("_cas","_fixmatch"))
        for _,row in joined.iterrows():
            rec={"label_fraction":frac,"seed":int(row["seed"])}
            for metric in available:
                rec[metric+"_cas_minus_fixmatch"]=float(row[metric+"_cas"]-row[metric+"_fixmatch"])
            primary_rows.append(rec)
    primary_paired=pd.DataFrame(primary_rows)
    print("Primary CAS-Match vs FixMatch paired deltas, seeds 42-46:")
    display(primary_paired.sort_values(["label_fraction","seed"]))
    if len(primary_paired):
        display(primary_paired.groupby("label_fraction")[
            [m+"_cas_minus_fixmatch" for m in available]].agg(["mean","std","count"]))
        for frac,group in primary_paired.groupby("label_fraction"):
            col="macro_f1_cas_minus_fixmatch"
            print(f"PRIMARY {frac:.0%} CAS-Match vs FixMatch: "
                  f"mean delta={group[col].mean()*100:.2f} pp; "
                  f"seed deltas={(group[col]*100).round(2).tolist()} | "
                  f"{classify_macro_f1_delta(group[col])}")
else:
    print("No completed runs yet.")


## Important limitations

This is a CRIC-specific study. Three seeds for control arms and five seeds for the primary FixMatch/CAS-Match comparison provide modest replication, not definitive statistical power. Strong-augmentation supervised control helps separate augmentation benefits from unlabeled-data benefits. The 100%-label supervised arm is a reference ceiling, not a matched low-label comparison.


## Output files

Output is isolated in `cric_final_protocol_v3/`. Each run folder retains `protocol.json`, `history.csv`, `pseudo_label_epoch_diagnostics.csv`, `test_probabilities.csv`, `test_metrics.json`, and `DONE.json`. Student/EMA/optimizer checkpoints are deleted after successful final evaluation to control disk use. Incomplete runs retain `checkpoint_latest.pt`, its `.previous` backup when available, and `checkpoint_best.pt`. The root summary is `cric_final_protocol_summary.csv`; split and subset hashes are in `protocol_audits/`.
